In [36]:
import pandas as pd
import numpy as np
from pathlib import Path 

frequency = 64
path = Path(f"data/raw/dreamt/data_{frequency}Hz")

COLS_TO_DROP = [
    "TIMESTAMP",
    "IBI",
    "Obstructive_Apnea",
    "Central_Apnea",
    "Hypopnea",
    "Multiple_Events",
]
nb_users_max = 25
X_all_patients = []
y_all_patients = []
patient_file_list = [f for f in path.iterdir() if f.is_file()]
for patient_id in range(nb_users_max):
    patient_file = patient_file_list.pop() 
    df = pd.read_csv(patient_file)
    df["Sleep_Stage"] = df["Sleep_Stage"].replace("P", "W")
    df = df.drop(
                columns=COLS_TO_DROP
            )
    df = df[df["Sleep_Stage"] != "Missing"]
    y_all_patients.append(df.Sleep_Stage.to_numpy())
    X_all_patients.append(df.drop(columns=["Sleep_Stage"]).to_numpy())
    df["patient_id"] = patient_id + 1 



In [48]:
WINDOWS_SEC = 30
FS = 64

window_samples = FS * WINDOWS_SEC

X_bvp = []
X_acc = []
X_eda_temp = []
X_hr = []
y = []

for patient in range(len(X_all_patients)):
    data = X_all_patients[patient]
    T = data.shape[0]
    n_windows = T // window_samples
    for i in range(n_windows):
        start = i * window_samples
        end = start + window_samples
        # 1920, 
        X_bvp.append(data[start:end,0])
        # 960
        X_acc.append(data[start:end:2, 1:4])
        # 120
        X_eda_temp.append(data[start:end:16, 4:6])
        # 30
        X_hr.append(data[start:end:64, 6])
        #1
        y.append(y_all_patients[patient][start])
        


In [74]:
X_bvp_whole =np.stack(X_bvp)
X_acc_whole =np.stack(X_acc)
X_eda_temp_whole =np.stack(X_eda_temp)
X_hr_whole =np.stack(X_hr)
y_whole = np.array(y)

In [75]:
print(X_bvp_whole.shape)
X_bvp_whole = np.expand_dims(X_bvp_whole, axis=1)
print(X_eda_temp_whole.shape)
X_eda_temp_whole = np.permute_dims(X_eda_temp_whole, axes=[0,2,1])
print(X_acc_whole.shape)
X_acc_whole = np.permute_dims(X_acc_whole, axes=[0,2,1])


(26882, 1920)
(26882, 120, 2)
(26882, 960, 3)


In [76]:
rng = np.random.default_rng(seed=64)
idx = np.arange(X_acc_whole.shape[0])
idx = rng.permutation(idx)

In [77]:
test_size = 0.2
dataset_len = X_bvp_whole.shape[0]

X_bvp_train = X_bvp_whole[idx[int(dataset_len * test_size) :]]
X_bvp_test = X_bvp_whole[idx[: int(dataset_len * test_size)]]

X_acc_train = X_acc_whole[idx[int(dataset_len * test_size) :]]
X_acc_test = X_acc_whole[idx[: int(dataset_len * test_size)]]

X_eda_temp_train = X_eda_temp_whole[idx[int(dataset_len * test_size) :]]
X_eda_temp_test = X_eda_temp_whole[idx[: int(dataset_len * test_size)]]

X_hr_train = X_hr_whole[idx[int(dataset_len * test_size) :]]
X_hr_test = X_hr_whole[idx[: int(dataset_len * test_size)]]

y_train = y_whole[idx[int(dataset_len * test_size) :]]
y_test = y_whole[idx[: int(dataset_len * test_size)]]


In [78]:
from sklearn.preprocessing import LabelEncoder

lb = LabelEncoder()
y_train_encoded = lb.fit_transform(y_train)
y_test_encoded = lb.transform(y_test)

y_train_encoded

array([4, 4, 4, ..., 2, 1, 1], shape=(21506,))

In [92]:
import torch
import torch.nn as nn


class MultiScaleCNN(nn.Module):
    def __init__(self):
        super().__init__()
        
        
        # input (B, 1, 1920)
        self.bvp_path = nn.Sequential(
        nn.Conv1d(in_channels = 1, out_channels = 32, kernel_size = 3, stride = 2, padding=1),
        nn.BatchNorm1d(32),
        nn.ReLU(),
        nn.MaxPool1d(2),
        # ,,480
        nn.Conv1d(in_channels = 32, out_channels = 64, kernel_size = 5, stride = 2, padding=2),
        nn.BatchNorm1d(64),
        nn.ReLU(),
        nn.MaxPool1d(2),
        # ,,120
        nn.Conv1d(in_channels = 64, out_channels = 128, kernel_size = 7, stride = 2, padding=3),
        nn.BatchNorm1d(128),
        nn.ReLU(),
        nn.MaxPool1d(2),
        ) # output B, 128, 30

        # input (B, 3, 960)
        self.acc_path = nn.Sequential(
        nn.Conv1d(in_channels=3 , out_channels= 32, kernel_size = 3, stride = 2, padding=1),
        nn.BatchNorm1d(32),
        nn.ReLU(),
        nn.MaxPool1d(2),
        # ,,240
        nn.Conv1d(in_channels=32 , out_channels= 64, kernel_size = 5, stride = 2, padding=2),
        nn.BatchNorm1d(64),
        nn.ReLU(),
        nn.MaxPool1d(4),
        ) # ,64, 30 

        self.eda_temp_path = nn.Sequential(
            nn.Conv1d(2, 16, kernel_size=3, stride=2, padding=1),  # 120→60
            nn.BatchNorm1d(16),
            nn.ReLU(),
            nn.MaxPool1d(2),                                        # 60→30
        )  # (B, 16, 30)
        
        self.fc = nn.Sequential(
            nn.LazyLinear(256),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(256, out_features=5),
        )

        self.init_weights()

    def init_weights(self):
       for layer in self.modules():
        if isinstance(layer, nn.Conv1d) or isinstance(layer, nn.Linear):
            if hasattr(layer, 'weight') and not isinstance(layer, nn.modules.lazy.LazyModuleMixin):
                nn.init.kaiming_normal_(layer.weight)
                nn.init.zeros_(layer.bias)
            
    def forward(self, x_bvp, x_acc, x_eda_temp, x_hr):
        out_bvp = self.bvp_path(x_bvp).flatten(1)
        out_acc = self.acc_path(x_acc).flatten(1)
        out_eda_temp = self.eda_temp_path(x_eda_temp).flatten(1)
        merged = torch.cat([out_bvp, out_acc, out_eda_temp, x_hr.flatten(1)], dim=1)
        return self.fc(merged)


In [81]:
from torch.utils.data import Dataset, DataLoader
class DreamtDataset(Dataset):
    def __init__(self, X_bvp, X_acc, X_eda_temp, X_hr, y):
        super().__init__()
        self.X_bvp      = X_bvp
        self.X_acc      = X_acc
        self.X_eda_temp = X_eda_temp
        self.X_hr       = X_hr
        self.y          = y

    def __getitem__(self, index):
        return (
            self.X_bvp[index],
            self.X_acc[index],
            self.X_eda_temp[index],
            self.X_hr[index],
            self.y[index],
        )

    def __len__(self):
        return len(self.X_bvp)



X_bvp_train      = torch.FloatTensor(X_bvp_train)
X_acc_train      = torch.FloatTensor(X_acc_train)
X_eda_temp_train = torch.FloatTensor(X_eda_temp_train)
X_hr_train       = torch.FloatTensor(X_hr_train)
y_train_encoded  = torch.LongTensor(y_train_encoded)  

train_ds = DreamtDataset(X_bvp_train, X_acc_train, X_eda_temp_train, X_hr_train, y_train_encoded)
train_dl = DataLoader(train_ds, batch_size=128, shuffle=True)

X_bvp_test       = torch.FloatTensor(X_bvp_test)
X_acc_test       = torch.FloatTensor(X_acc_test)
X_eda_temp_test  = torch.FloatTensor(X_eda_temp_test)
X_hr_test        = torch.FloatTensor(X_hr_test)
y_test_encoded   = torch.LongTensor(y_test_encoded)

test_ds = DreamtDataset(X_bvp_test, X_acc_test, X_eda_temp_test, X_hr_test, y_test_encoded)
test_dl = DataLoader(test_ds, batch_size=1024)

In [109]:
from sklearn.utils.class_weight import compute_class_weight

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

classes = np.unique(y_train_encoded.numpy())
weights = compute_class_weight(class_weight="balanced", classes=classes, y=y_train_encoded.numpy())
weights = torch.FloatTensor(weights).to(DEVICE)

criterion = nn.CrossEntropyLoss(weight=weights, reduction="sum")

In [110]:
from tqdm import tqdm


def train_model(model, train_dl, epochs, weights= None, lr=0.001, device=torch.device("cpu")):
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    if not weights:
        criterion = nn.CrossEntropyLoss(reduction="sum")
    else: 
        criterion = nn.CrossEntropyLoss(weight=weights, reduction="sum")
    for epoch in tqdm(range(epochs)):
        model.train()
        empirical_risk = 0.0
        for x_bvp, x_acc, x_eda_temp, x_hr, y in train_dl:
            x_bvp = x_bvp.to(device)
            x_acc = x_acc.to(device)
            x_eda_temp = x_eda_temp.to(device)
            x_hr = x_hr.to(device)
            y = y.to(device)
            optimizer.zero_grad()
            pred = model(x_bvp, x_acc, x_eda_temp, x_hr)
            loss = criterion(pred, y)
            loss.backward()
            optimizer.step()
            empirical_risk += loss.item()

        empirical_risk /= len(train_dl.dataset)
        print("Train loss: %.3f" % (empirical_risk))


model = MultiScaleCNN()
model.to(DEVICE)

train_model(model, train_dl, epochs=50,device = DEVICE)


  0%|          | 0/50 [00:00<?, ?it/s]

  2%|▏         | 1/50 [00:07<05:52,  7.19s/it]

Train loss: 1.498


  4%|▍         | 2/50 [00:14<05:42,  7.13s/it]

Train loss: 1.072


...

Train loss: 0.694


 98%|█████████▊| 49/50 [05:47<00:07,  7.07s/it]

Train loss: 0.703


100%|██████████| 50/50 [05:54<00:00,  7.09s/it]

Train loss: 0.699


In [111]:
def test_model(model, test_dl, device=torch.device("cpu")):
    criterion = nn.CrossEntropyLoss(reduction="sum")
    model.eval()
    generalization_error = 0.0
    correct = 0
    all_preds = []
    all_targets = []

    with torch.no_grad():
        for x_bvp, x_acc, x_eda_temp, x_hr, y in test_dl:
            x_bvp = x_bvp.to(device)
            x_acc = x_acc.to(device)
            x_eda_temp = x_eda_temp.to(device)
            x_hr = x_hr.to(device)
            y = y.to(device)
            logits = model(x_bvp, x_acc, x_eda_temp, x_hr)
            loss = criterion(logits, y)
            pred = torch.argmax(logits, dim=1)
            correct += (pred == y).sum().item()
            generalization_error += loss.item()
            all_preds.append(pred.cpu())
            all_targets.append(y.cpu())

        y_pred = torch.cat(all_preds).numpy()
        y_true = torch.cat(all_targets).numpy()
        generalization_error /= len(test_dl.dataset)
        accuracy = correct / len(test_dl.dataset)
        print(
            "Generalization Error: %.3f, Accuracy %.3f"
            % (generalization_error, accuracy)
        )

    return y_true, y_pred


y_true, y_pred = test_model(model, test_dl, DEVICE)


Generalization Error: 0.925, Accuracy 0.705


In [ ]:
from sklearn.metrics import f1_score, classification_report

f1_score(y_true, y_pred, average="macro")

print(classification_report(y_true, y_pred, target_names=["N1","N2","N3","R","W"]))

0.3995006615959008
              precision    recall  f1-score   support

          N1       0.00      0.00      0.00       369
          N2       0.61      0.89      0.73      1980
          N3       0.78      0.18      0.29       222
           R       0.52      0.10      0.16       451
           W       0.82      0.83      0.82      2354

    accuracy                           0.70      5376
   macro avg       0.55      0.40      0.40      5376
weighted avg       0.66      0.70      0.65      5376



/home/yacine/sleep-stage-prediction/.venv/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/yacine/sleep-stage-prediction/.venv/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/yacine/sleep-stage-prediction/.venv/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, 

In [105]:
np.argwhere(y_test == "N2").shape

(1980, 1)